In [1]:
l1 = ["zx", "zy", "yx"] 
l2 = ["cab", "cad", "bd", "bc"] 
l3 = ["abc", "ab"] 
l4 = ["abc"] 

In [ ]:
def question_1(palavras: list) -> str:
    """
    Alfabeto Perdido.

    palavras: lista de palavras (letras 'a'..'z', não vazias) em ordem
              lexicográfica segundo uma ordem de letras DESCONHECIDA.
              1 <= len(palavras) <= 10^5; o alfabeto tem no máximo 26 letras;
              a lista pode conter palavras repetidas.

    Retorna uma string com cada letra distinta que aparece nas palavras
    exatamente uma vez, em uma ordem consistente com a ordenação observada
    (qualquer ordem válida é aceita). Se a lista for inconsistente, retorna "".
    Uma palavra nunca pode vir antes de um prefixo próprio seu
    (ex: "abc" antes de "ab" é inconsistente).

    Complexidade exigida: O(C), C = total de caracteres da lista.

    RESPOSTA:

    Dada a sequência de palavras, criamos o grafo com base nela comparando cada dupla
    de palavras em sequência e verificando a ordem aparente dos caracteres, em que cada 
    nó é um caractere e existe uma aresta direcionada entre os caracteres da ordem vista 
    na sequência. Para que haja uma ordem lexicográfica válida, precisamos que o grafo 
    admita uma ordem topológica.

    Para isso, criamos a função create_graph que retorna o grafo encontrado na sequência
    implementado como um dicionário python em que cada chave é um nó e cada valor um conjunto 
    dos nós que recebem uma aresta que parte do nó chave. Para auxiliar o algoritmo que verifica 
    a ordem lexicográfica, a função também retorna um dicionário que armazena o grau de 
    entrada de cada vértice. Caso todos os caracteres da primeira palavra sejam comparados 
    com o da segunda, todos sejam iguais e a segunda ainda tenha caracteres a serem comparados,
    então retornamos uma string vazia, porque um prefixo de uma palavra veio depois dela.

    A partir disso a implementação do algoritmo segue o seguinte: mantemos uma fila com 
    todos os nós que tem grau de entrada igual a 0, que são candidatos a serem nós fontes.
    Para cada um deles, estabelecemos sua posição na ordem e verificamos todos os seus vértices
    adjacentes e subtraimos 1 de seus graus de entrada, adicionando à fila os vértices com 
    grau de entrada igual a 0. Se todos os vértices forem retirados retornamos a ordem. 
    Caso contrário, retornamos uma string vazia.

    """

    def create_graph(lista: list[str]):
        graph = {ch: set() for word in lista for ch in word}
        in_Degree = {ch: 0 for ch in graph}

        for w1, w2 in zip(lista, lista[1:]):
            for ch1, ch2 in zip(w1, w2):
                if ch1 != ch2:
                    if ch2 not in graph[ch1]:
                        graph[ch1].add(ch2)
                        in_Degree[ch2] += 1
                    break
            else:
                if len(w1) > len(w2):
                    return None, None

        return graph, in_Degree

    def has_topological_order(graph: dict, in_Degree: dict):
        if graph == None: return ""
        queue = [v for v, deg in in_Degree.items() if deg == 0]
        head = 0
        order = ""

        while head < len(queue):
            v = queue[head]
            head += 1
            order = order + v
            for v_next in graph[v]:
                in_Degree[v_next] -= 1
                if in_Degree[v_next] == 0:
                    queue.append(v_next)

        if len(order) == len(graph): return order
        else: return ""

    graph, in_Degree = create_graph(palavras)
    return has_topological_order(graph, in_Degree)
    


In [ ]:
Entrada1 =  [[1, 1, 0, 0, 0],
            [1, 0, 0, 0, 0],
            [0, 0, 0, 0, 0],
            [0, 0, 0, 1, 1],
            [0, 0, 0, 1, 1]]

Entrada2 = [[1, 0, 0, 0, 1],
            [1, 0, 0, 0, 1],
            [1, 0, 0, 0, 1]]

Entrada3 = [[0, 1, 0],
            [0, 0, 0],
            [0, 0, 1]]

def question_2(mapa: list) -> int:
    """
    Ponte entre as Ilhas.

    mapa: matriz n x m (lista de listas) de 0 (água) e 1 (terra), com
          2 <= n, m <= 500. Há exatamente duas ilhas (conjuntos maximais de
          células de terra conectadas na vertical/horizontal).

    Retorna o menor número de células de água que precisam ser convertidas
    em terra para que as duas ilhas formem uma só.

    Complexidade exigida: O(n · m).

    RESPOSTA:

    A ideia aqui é utilizar algumas informações do problema a nosso favor. 
    Sabemos que há apenas duas ilhas. Dessa forma, vamos prosseguir da seguinte forma:

    Primeiro, começando do canto superior esquerdo do grid, a entrada (0,0), vamos buscar 
    a primeira entrada com valor 1, assim que encontrarmos, vamos alterar o valor das entradas 
    em sua adjacência que também tem valor igual a 1 para marcar a primeira ilha. Faremos 
    isso implementando BFS, onde cada vértice é uma entrada e as arestas conectam cada entrada 
    à sua adjacência.

    Depois que separamos a ilha 1 e ilha 2, vamos fazer a BFS a partir da ilha 1 para conseguir
    a menor distância até a ilha 2.

    """

    def breadth_first_modify(matrix, pos, searched, flag):
        m, n = len(matrix), len(matrix[0])
        queue = [pos]
        if matrix[pos[0]][pos[1]] == searched:
            matrix[pos[0]][pos[1]] = flag
        else:
            return matrix, queue
        
        visited = {pos}
        head = 0
        
        while head < len(queue):
            v = queue[head]
            head += 1
            adjacency = [(v[0] - 1, v[1]), (v[0], v[1] - 1), (v[0] + 1, v[1]), (v[0], v[1] + 1)]
            for i, j in adjacency:
                if 0 <= i < m and 0 <= j < n and (i,j) not in visited:
                    if matrix[i][j] == searched: 
                        matrix[i][j] = flag
                        visited.add((i,j))
                        queue.append((i, j))

        return matrix, queue

    def breadth_first_search(matrix, starts, searched):
        m, n = len(matrix), len(matrix[0])

        for pos in starts:
            if matrix[pos[0]][pos[1]] == searched:
                return pos, 0
        
        queue = [(pos, 0) for pos in starts]
        visited = set(starts)
        head = 0
        
        while head < len(queue):
            v, dist = queue[head]
            head += 1
            adjacency = [(v[0] - 1, v[1]), (v[0], v[1] - 1), (v[0] + 1, v[1]), (v[0], v[1] + 1)]
            for i, j in adjacency:
                if 0 <= i < m and 0 <= j < n and (i,j) not in visited:
                    if matrix[i][j] == searched: 
                        return (i,j), dist 
                    visited.add((i,j))
                    queue.append(((i, j), dist + 1))

        return (None, None), -1

    pos, _ = breadth_first_search(mapa, [(0, 0)], 1)
    matrix, queue = breadth_first_modify(mapa, pos, 1, 2)
    _, menor = breadth_first_search(matrix, queue, 1)

    return menor


question_2(Entrada1)





    


    



4

In [ ]:
def question_3(altitudes: list) -> tuple:
    """
    Trilha de Menor Esforço.

    altitudes: matriz n x m (lista de listas) de inteiros,
               1 <= n, m <= 300, 1 <= altitude <= 10^6.
    A caminhante parte de (0, 0) e chega em (n-1, m-1), movendo-se para uma das
    4 células vizinhas a cada passo. O esforço de uma trilha é a maior
    diferença absoluta de altitude entre duas células consecutivas dela.

    Retorna (esforco, passos): o menor esforço possível entre todas as trilhas
    e, entre as trilhas que atingem esse esforço, o menor número de passos.

    Complexidade exigida: O(n · m · log(n · m)) ou melhor.

    RESPOSTA:

    A ideia aqui é parecida com a última questão com relação aos nós e vértices.
    Vamos considerar cada entrada como um nó, e há uma aresta conectando o nó às
    suas entradas adjacentes, com peso igual ao valor absoluto da diferença entre o valor
    das entradas.

    Dessa forma, como queremos saber qual caminho a trilha de menor esforço da origem (0,0)
    até (n-1,m-1). Dessa forma, a ideia foi implementar Dijkstra, como não temos pesos negativos,
    e invés de operar somando os pesos, atualizar os vértices usando o máximo entre o esforço do vértice 
    anterior e posterior, armazenando também a quantidade de passos.

    Definimos target e source, e inicializamos todos os esforços como infinitos, a menos do source,
    que definimos como (esforço= 0, passos= 0) e inserimos ele dentro de uma MinHeap. A partir disso,
    começamos retirando o primeiro elemento da Heap e definimos ele como v, verificamos se ele é o target, 
    se for, retornamos seu esforço e passos, caso contrário, continuamos. Para cada vértice adjacente
    a v, definimos novo_esforco como o máximo entre o esforço do vértice anterior e novos_passos como 
    passos até o vértice anterior + 1, se o novo_esforco for menor que o esforço atual para chegar até
    o vértice hoje, atualizamos o valor e inserimos elena heap, caso contrário continuamos.

    """


    class MinHeap():
        def __init__(self):
            self.data = []
            self.pos = {}

        def __len__(self): return len(self.data)

        def swap(self, i, j):
            self.data[i], self.data[j] = self.data[j], self.data[i]
            self.pos[self.data[i][1]] = i
            self.pos[self.data[j][1]] = j

        def heapify_up(self, idx):
            while idx > 0:
                pai = (idx - 1)//2
                if self.data[pai] > self.data[idx]:
                    self.swap(pai, idx)
                    idx = pai
                else:
                    break

        def heapify_down(self, idx):
            while idx * 2 + 1 < len(self.data):
                l = idx * 2 + 1
                r = idx * 2 + 2
                menor = l
                if r < len(self.data) and self.data[r] < self.data[menor]:
                    menor = r
                if self.data[menor] < self.data[idx]:
                    self.swap(menor, idx)
                    idx = menor
                else:
                    break

        def insert_or_update(self, node):
            v, w = node[0], node[1]

            if v in self.pos:
                idx = self.pos[v]
                if w < self.data[self.pos[v]][0] :
                    self.data[idx] = (w, v)
                    self.heapify_up(idx)

            else:
                self.data.append((w, v))
                idx = len(self.data) - 1
                self.pos[v] = idx
                self.heapify_up(idx)

        def pop(self):
            node = self.data[0]
            self.data[0] = self.data[-1]
            self.data.pop()
            del self.pos[node[1]]
            if self.data:
                self.pos[self.data[0][1]] = 0
                self.heapify_down(0)

            return node


    def Dijkstra(map, source):
        m, n = len(map), len(map[0])
        target = (m-1, n-1)

        best = {(i,j): (float('inf'), float('inf')) for i in range(m) for j in range(n)}
        best[source] = (0,0)

        heap = MinHeap()
        heap.insert_or_update((source, (0, 0)))
        

        while heap:

            (esforco, passos), pos = heap.pop()
        
            if pos == target: 
                return (esforco, passos)

            adjacency = [(pos[0]-1, pos[1]), (pos[0], pos[1]-1), (pos[0]+1, pos[1]), (pos[0], pos[1]+1)]

            for i,j in adjacency:

                if 0 <= i < m and 0 <= j < n:
                    w = abs(map[i][j] - map[pos[0]][pos[1]])
                    novo_esforco = max(esforco, w)
                    novos_passos = passos + 1
                    novo_estado = (novo_esforco, novos_passos)

                    if novo_estado < best[(i,j)]:
                        best[(i,j)] = novo_estado
                        heap.insert_or_update(((i,j), novo_estado))

        return best[target]

    return Dijkstra(altitudes, (0, 0))





In [ ]:
import math

def question_4(n: int, cotacoes: list) -> list:
    """
    Câmbio Ineficiente.

    n: número de moedas (0..n-1), 2 <= n <= 300.
    cotacoes: lista de tuplas (a, b, r), com a != b: 1 unidade da moeda a vale r
              unidades da moeda b (só nesse sentido). No máximo uma cotação por
              par ordenado.
              1 <= len(cotacoes) <= 5000, 10^-3 <= r <= 10^3.

    Arbitragem: sequência de trocas que começa e termina na mesma moeda com
    estritamente mais do que se começou (produto das cotações do ciclo > 1,
    com tolerância de 1e-9: ciclos de produto 1 NÃO são arbitragem).

    Retorna uma lista [c0, c1, ..., ck] com ck == c0 descrevendo UM ciclo de
    arbitragem, ou [] se não existir nenhum.

    Complexidade exigida: O(n · m), m = len(cotacoes).

    RESPOSTA:

    Não cheguei a ideia de usar -log sozinho para transformar as multiplicações em soma. 
    Porém, depois de conversar com colegas sobre essa possibilidade, a implementação do 
    algoritmo foi minha.

    A ideia principal foi que 
    
    r1*r2*r3... > 1 -> sum(log(r)) > 0 -> -sum(log(r)) < 0

    E, portanto, o problema vira um problema de encontrar um ciclo negativo. Utilizamos então
    Bellman-Ford para encontrar se há ou não. 

    """
    
    def bellman_ford(sequence, n= n):
        graph = [[] for _ in range(n)]
        for v, v_next, r in sequence:
            graph[v].append((v_next, -math.log(r), r))

        distances = [0.0 for _ in range(n)]
        parents = [-1 for _ in range(n)]
    

        for _ in range(n):
            for v in range(n):
                for v_next, w, r in graph[v]:
                    if distances[v] + w < distances[v_next]:
                        distances[v_next] = distances[v] + w
                        parents[v_next] = v
                        print(distances)

        for v in range(n):
            for v_next, w, r in graph[v]:
                if distances[v] + w < distances[v_next]:
                    parents[v_next] = v

                    x = v_next
                    for _ in range(n):
                        x = parents[x]

                    cycle = [x]
                    pai = parents[x]
                    while pai != x:
                        cycle.append(pai)
                        pai = parents[pai]
                    cycle.append(x)
                    cycle.reverse()

                    print(cycle)
                    print(graph)

                    vrf = 1
                    for v, v_other in zip(cycle, cycle[1:]):
                        for v_next, w, r in graph[v]:
                            if v_next == v_other:
                                vrf = vrf * r

                    return cycle if vrf > 1.0 + 1e-9 else []
                        
        return []

    bellman_ford(cotacoes)


[0.0, -0.6931471805599453, 0.0]
[0.0, -0.6931471805599453, -1.791759469228055]
[-0.18232155679395468, -0.6931471805599453, -1.791759469228055]
[-0.18232155679395468, -0.8754687373539, -1.791759469228055]
[-0.18232155679395468, -0.8754687373539, -1.9740810260220099]
[-0.3646431135879096, -0.8754687373539, -1.9740810260220099]
[-0.3646431135879096, -1.0577902941478547, -1.9740810260220099]
[-0.3646431135879096, -1.0577902941478547, -2.1564025828159643]
[-0.546964670381864, -1.0577902941478547, -2.1564025828159643]
[1, 2, 0, 1]
[[(1, -0.6931471805599453, 2.0)], [(2, -1.0986122886681098, 3.0)], [(0, 1.6094379124341003, 0.2)]]


In [30]:
def question_5(n: int, ligacoes: list, consultas: list) -> list:
    """
    Organograma da Empresa.

    n: número de funcionários (0..n-1), raiz da hierarquia é o 0.
       2 <= n <= 2 * 10^5. A hierarquia pode ser uma "cadeia".
    ligacoes: n-1 pares (u, v) que indicam que u e v estão diretamente ligados
              (não indicam quem é o chefe de quem).
    consultas: lista de q tuplas (tipo, u, v), 1 <= q <= 2 * 10^5:
        ("A", u, v): u é chefe (direto ou indireto) de v?
        ("P", u, v): u e v são primos (mesmo nível, chefes diretos diferentes)?
    Em ambas as consultas, se u == v a resposta é False.

    Retorna a lista de respostas (bool) na ordem das consultas.

    Complexidade exigida: O(n + q) no total (cada consulta O(1) após o
    pré-processamento).

    RESPOSTA:

    A ideia aqui é de implementar o algoritmo usando a lógica de intervalo de vida
    que o DFS gera, assim um v1 é ancestral de v2 se pre_order[v1] < pre_order[v2]
    e post_order[v1] > post_order[v2], dado que v1 é visitado antes de v2 e é exaurido depois 
    de v2. E v1 é primo de v2 se estiverem na mesma altura, mas tiverem pais diferentes.

    Para isso, armazenamos, pre_order, post_order, parents e depth.

    Depois de retornar esses vetores, conseguimos fazer cada consulta em O(1)
    porque elas operam apenas sobre índices.

    """

    graph = [[] for _ in range(n)]
    for v, v_next in ligacoes:
        graph[v].append(v_next)
        graph[v_next].append(v)

    def depth_first_search(graph, source):
        stack = [(source, 0)]

        pre_order = [-1 for _ in range(len(graph))]
        count_pre = 0
        post_order = [-1 for _ in range(len(graph))]
        count_post = 0

        parents = [-1 for _ in range(len(graph))]

        depth = [-1 for _ in range(len(graph))]
        depth[source] = 0

        while stack:
            v, flag = stack.pop()

            if flag == 0:
                pre_order[v] = count_pre
                count_pre += 1

                stack.append((v, 1))
            

                for v_next in graph[v]:
                    if depth[v_next] == -1:
                        stack.append((v_next, 0))
                        depth[v_next] = depth[v] + 1
                        parents[v_next] = v

            else:
                post_order[v] = count_post
                count_post += 1


        return pre_order, post_order, depth, parents

    def consultar(consulta, pre_order, post_order, depth, parents):
        flag, v1, v2 = consulta

        if flag == "A":
            return (v1 != v2 and pre_order[v1] < pre_order[v2] and post_order[v1] > post_order[v2])

        if flag == "P":
            return (v1 != v2 and depth[v1] == depth[v2] and parents[v1] != parents[v2])

    pre_order, post_order, depth, parents = depth_first_search(graph, 0)
    results = [consultar(consulta, pre_order, post_order, depth, parents) for consulta in consultas] 
    return results

n = 7 
ligacoes = [(0,1), (0,2), (1,3), (1,4), (2,5), (2,6)]
consultas = [("A", 0, 5), ("A", 1, 5), ("A", 3, 3), ("P", 3, 5), ("P", 3, 4), ("P", 1, 3)]

question_5(n, ligacoes, consultas)




[True, False, False, True, False, False]

In [ ]:
def question_6(pocos: list, canos: list) -> int:
    """
    Abastecimento de Água.

    pocos: lista de n custos; pocos[i] é o custo de cavar um poço na casa i.
    canos: lista de tuplas (i, j, custo), com i != j e no máximo um cano por par
           de casas: cano possível entre as casas i e j (funciona nos dois
           sentidos; a água é transmitida por toda sequência de canos ligados).
    1 <= n <= 10^4, 0 <= len(canos) <= 10^5, 1 <= custos <= 10^5.

    Retorna o menor custo total para abastecer todas as casas.

    Complexidade exigida: O((n + m) log n), m = len(canos).

    RESPOSTA:

    Queremos implementar um algoritmo que encontre 
    """

    class MinHeap():
        def __init__(self):
            self.data = []
            self.pos = {}

        def __len__(self): return len(self.data)

        def swap(self, i, j):
            self.data[i], self.data[j] = self.data[j], self.data[i]
            self.pos[self.data[i][1]] = i
            self.pos[self.data[j][1]] = j

        def heapify_up(self, idx):
            while idx > 0:
                pai = (idx - 1)//2
                if self.data[pai] > self.data[idx]:
                    self.swap(pai, idx)
                    idx = pai
                else:
                    break

        def heapify_down(self, idx):
            while idx * 2 + 1 < len(self.data):
                l = idx * 2 + 1
                r = idx * 2 + 2
                menor = l
                if r < len(self.data) and self.data[r] < self.data[menor]:
                    menor = r
                if self.data[menor] < self.data[idx]:
                    self.swap(menor, idx)
                    idx = menor
                else:
                    break

        def insert_or_update(self, node):
            v, w = node[0], node[1]

            if v in self.pos:
                idx = self.pos[v]
                if w < self.data[self.pos[v]][0] :
                    self.data[idx] = (w, v)
                    self.heapify_up(idx)

            else:
                self.data.append((w, v))
                idx = len(self.data) - 1
                self.pos[v] = idx
                self.heapify_up(idx)

        def pop(self):
            node = self.data[0]
            self.data[0] = self.data[-1]
            self.data.pop()
            del self.pos[node[1]]
            if self.data:
                self.pos[self.data[0][1]] = 0
                self.heapify_down(0)

            return node

    def Prim_modified(pocos, canos):
        V = len(pocos)
        graph = [[] for _ in range(V)]
        for v, v_next, w in canos:
            graph[v].append((v_next, w))
            graph[v_next].append((v, w))
        
        in_Tree = [False for _ in range(V)]

        heap = MinHeap()
        for idx in range(V):
            heap.insert_or_update((idx, pocos[idx]))

        total_cost = 0

        while heap:
            cost, v = heap.pop()
            print(v)
            in_Tree[v] = True
            
            total_cost += cost
            print(f"Vértice colocado = {v}, custo total = {total_cost}")

            for v_next, w in graph[v]:
                if not in_Tree[v_next]: 
                    cost = w
                    heap.insert_or_update((v_next, cost))

        return total_cost

    total_cost = Prim_modified(pocos, canos)
    return total_cost

# Exemplo 1 (saída esperada: 10)
pocos = [5, 4, 6, 10]
canos = [(0, 1, 1), (1, 2, 2), (2, 3, 3), (0, 3, 8)]
print(question_6(pocos, canos))

# Exemplo 2 (saída esperada: 3)
pocos = [1, 2, 2]
canos = [(0, 1, 1), (1, 2, 1)]
print(question_6(pocos, canos))

        

1
Vértice colocado = 1, custo total = 4
0
Vértice colocado = 0, custo total = 5
2
Vértice colocado = 2, custo total = 7
3
Vértice colocado = 3, custo total = 10
10
0
Vértice colocado = 0, custo total = 1
1
Vértice colocado = 1, custo total = 2
2
Vértice colocado = 2, custo total = 3
3


In [ ]:
relevo = [[1,2,3],[2,3,4],[3,4,5]] 

def question_7(relevo: list) -> tuple:
    """
    Trilha de Escalada.

    relevo: matriz n x m (lista de listas) de altitudes,
            1 <= n, m <= 300, 0 <= altitude <= 10^9 (valores podem se repetir).
    Uma escalada começa em qualquer célula e a cada passo move-se para uma das
    4 vizinhas de altitude ESTRITAMENTE maior. Pode ter uma única célula.

    Retorna (L, c): L é o maior número de células de uma escalada e c é a
    quantidade de escaladas distintas com exatamente L células, módulo 10^9 + 7.
    (Duas escaladas são distintas se diferem na sequência de células visitadas.)

    Complexidade exigida: O(n · m) ou O(n · m · log(n · m)).

    RESPOSTA:

    A ideia aqui é fazer a busca do maior para o menor. Iniciamos duas matrizes f e g, cada uma preenchida
    com uns. 
    
    Para cada entrada (i,j): 
    f[i][j] = Tamanho da maior escalada começando em (i,j) 
    g[i][j] = Quantas escaladas desse tamanho começam em (i,j)

    Ordenamos as entradas para serem processadas de forma decrescente por altura: custo 0(nmlog(nm)) para ordenar todas
    as entradas.

    Iteramos essa lista ordenada da seguinte forma: 

    Para cada entrada (altura, i, j) da lista:
    I)   Verificamos sua adjacência 

    II)  Selecionamos 'melhor' como o valor da maior trilha possível a partir de um vértice da adjacência e 'formas' como 
    o número de formas possíveis de realizar trilhas de tamanho 'melhor' a partir daquele vértice

    III) Se encontrarmos outro vértice da adjacência com tamanho da maior trilha a partir dele = 'melhor' já estabelecido,
    somamos ao 'formas' o número de formas possíveis de realizar trilhas de tamanho 'melhor' a partir desse outro vértice

    IV) Para o vértice (i,j): atualizamos o valor da maior trilha possível a partir daquele vértice como o maior valor da 
    maior trilha possível a partir dos vértices de sua adjacência (que é o 'melhor') + 1, dado que estamos adicionando o 
    vértice (i,j) a essa maior trilha e atualizamos o número de formas de realizar a trilha de tamanho 'melhor' a partir
    de (i,j) como a soma das formas possíveis a partir dos vértices de sua adjacência que que possuem f = 'melhor' (que é
    o 'formas') dado que adicionar mais um vértice a essas trilhas aumenta o número de vértices, mas não aumenta o número 
    de possibilidades.

    Por fim, encontramos L = a entrada com maior valor em f e c = somatório de todas as possibilidades de trilhas com tamanho
    L.
    """

    m, n = len(relevo), len(relevo[0])
    MOD = 10**9 + 7

    f = [[1] * n for _ in range(m)]
    g = [[1] * n for _ in range(m)]
 
    celulas = sorted(((relevo[i][j], i, j) for i in range(m) for j in range(n)), reverse=True)

    for alt, i, j in celulas:
        melhor = 0
        formas = 0

        passos = ((1,0), (0,1), (-1,0), (0,-1))
        for dx, dy in passos:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and relevo[x][y] > alt:
                if f[x][y] > melhor:
                    melhor = f[x][y]
                    formas = g[x][y]
                elif f[x][y] == melhor:
                    formas += g[x][y]

        if melhor > 0:
            f[i][j] = melhor + 1
            g[i][j] = formas % MOD

    L = max(max(linha) for linha in f)
    c = sum(g[i][j] for i in range(m) for j in range(n) if f[i][j] == L) % MOD
    return (L, c)




question_7(relevo)

    

(5, 6)